In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

In [0]:
customers_data = [
    (1, "John Doe", "john@example.com", "Hyderabad"),
    (2, "Alice ", "alice@example.com", "Chennai"),
    (3, None, "bob@example.com", "Bangalore"),        # NULL name
    (4, "David", None, "Mumbai"),                    # NULL email
    (5, "Eva", "eva@example.com", "Hyderabad"),
    (6, "Frank", "frank@example.com", "Delhi"),
]

customers = spark.createDataFrame(customers_data, ["customer_id", "name", "email", "city"])

In [0]:
orders_data = [
    (101, 1, "2024-01-01", 1000),
    (102, 2, "2024-01-02", 2000),
    (103, 3, "2024-01-03", -500),     # INVALID negative value
    (104, 99, "2024-01-04", 1500),    # INVALID FK (customer_id 99)
    (105, 1, "2024-01-05", None),     # NULL amount
    (106, 5, "2024-01-06", 3000),
    (107, 5, "2024-01-07", 3000),     # duplicate-like record
]

orders = spark.createDataFrame(orders_data, ["order_id", "customer_id", "order_date", "amount"])

In [0]:
orders = orders.withColumn("order_date", to_date(col("order_date")))

### Practice Set A: Join Drills

### 1. Inner Join (valid records only)

In [0]:
inner_df = orders.join(customers, on="customer_id", how="inner")
inner_df.show()

+-----------+--------+----------+------+--------+-----------------+---------+
|customer_id|order_id|order_date|amount|    name|            email|     city|
+-----------+--------+----------+------+--------+-----------------+---------+
|          1|     105|2024-01-05|  NULL|John Doe| john@example.com|Hyderabad|
|          1|     101|2024-01-01|  1000|John Doe| john@example.com|Hyderabad|
|          2|     102|2024-01-02|  2000|  Alice |alice@example.com|  Chennai|
|          3|     103|2024-01-03|  -500|    NULL|  bob@example.com|Bangalore|
|          5|     107|2024-01-07|  3000|     Eva|  eva@example.com|Hyderabad|
|          5|     106|2024-01-06|  3000|     Eva|  eva@example.com|Hyderabad|
+-----------+--------+----------+------+--------+-----------------+---------+



In [0]:
print("Inner Join Count:", inner_df.count())

Inner Join Count: 6


### 2. Left Join (identify nulls)

In [0]:
left_df = orders.join(customers, on="customer_id", how="left")
left_df.show()

+-----------+--------+----------+------+--------+-----------------+---------+
|customer_id|order_id|order_date|amount|    name|            email|     city|
+-----------+--------+----------+------+--------+-----------------+---------+
|          1|     101|2024-01-01|  1000|John Doe| john@example.com|Hyderabad|
|          2|     102|2024-01-02|  2000|  Alice |alice@example.com|  Chennai|
|          3|     103|2024-01-03|  -500|    NULL|  bob@example.com|Bangalore|
|         99|     104|2024-01-04|  1500|    NULL|             NULL|     NULL|
|          1|     105|2024-01-05|  NULL|John Doe| john@example.com|Hyderabad|
|          5|     106|2024-01-06|  3000|     Eva|  eva@example.com|Hyderabad|
|          5|     107|2024-01-07|  3000|     Eva|  eva@example.com|Hyderabad|
+-----------+--------+----------+------+--------+-----------------+---------+



### Find missing customer records:

In [0]:
invalid_fk_df = left_df.filter(col("name").isNull())
invalid_fk_df.show()

+-----------+--------+----------+------+----+---------------+---------+
|customer_id|order_id|order_date|amount|name|          email|     city|
+-----------+--------+----------+------+----+---------------+---------+
|          3|     103|2024-01-03|  -500|NULL|bob@example.com|Bangalore|
|         99|     104|2024-01-04|  1500|NULL|           NULL|     NULL|
+-----------+--------+----------+------+----+---------------+---------+



### Left Anti Join (invalid foreign keys)

In [0]:
invalid_orders = orders.join(customers, on="customer_id", how="left_anti")
invalid_orders.show()

+-----------+--------+----------+------+
|customer_id|order_id|order_date|amount|
+-----------+--------+----------+------+
|         99|     104|2024-01-04|  1500|
+-----------+--------+----------+------+



### Compare Row Counts

In [0]:
print("Orders Count:", orders.count())
print("Inner Join Count:", inner_df.count())
print("Left Join Count:", left_df.count())
print("Invalid FK Count:", invalid_orders.count())

Orders Count: 7
Inner Join Count: 6
Left Join Count: 7
Invalid FK Count: 1


### Practice Set B: Window Functions

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

### Top 3 Customers per City

In [0]:
window_city = Window.partitionBy("city").orderBy(col("amount").desc())
top_customers = inner_df.withColumn("rank", rank().over(window_city)).filter(col("rank") <= 3)
top_customers.show()

+-----------+--------+----------+------+--------+-----------------+---------+----+
|customer_id|order_id|order_date|amount|    name|            email|     city|rank|
+-----------+--------+----------+------+--------+-----------------+---------+----+
|          3|     103|2024-01-03|  -500|    NULL|  bob@example.com|Bangalore|   1|
|          2|     102|2024-01-02|  2000|  Alice |alice@example.com|  Chennai|   1|
|          5|     107|2024-01-07|  3000|     Eva|  eva@example.com|Hyderabad|   1|
|          5|     106|2024-01-06|  3000|     Eva|  eva@example.com|Hyderabad|   1|
|          1|     101|2024-01-01|  1000|John Doe| john@example.com|Hyderabad|   3|
+-----------+--------+----------+------+--------+-----------------+---------+----+



### Running Total of Sales

In [0]:
window_running = Window.orderBy("order_date")
running_total = inner_df.withColumn("running_total",sum("amount").over(window_running))
running_total.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+--------+----------+------+--------+-----------------+---------+-------------+
|customer_id|order_id|order_date|amount|    name|            email|     city|running_total|
+-----------+--------+----------+------+--------+-----------------+---------+-------------+
|          1|     101|2024-01-01|  1000|John Doe| john@example.com|Hyderabad|         1000|
|          2|     102|2024-01-02|  2000|  Alice |alice@example.com|  Chennai|         3000|
|          3|     103|2024-01-03|  -500|    NULL|  bob@example.com|Bangalore|         2500|
|          1|     105|2024-01-05|  NULL|John Doe| john@example.com|Hyderabad|         2500|
|          5|     106|2024-01-06|  3000|     Eva|  eva@example.com|Hyderabad|         5500|
|          5|     107|2024-01-07|  3000|     Eva|  eva@example.com|Hyderabad|         8500|
+-----------+--------+----------+------+--------+-----------------+---------+-------------+



### Rank Customers by Total Spend

In [0]:
total_spend = inner_df.groupBy("customer_id", "name").agg(sum("amount").alias("total_spend"))
window_rank = Window.orderBy(col("total_spend").desc())
ranked_customers = total_spend.withColumn("rank", dense_rank().over(window_rank))
ranked_customers.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+--------+-----------+----+
|customer_id|    name|total_spend|rank|
+-----------+--------+-----------+----+
|          5|     Eva|       6000|   1|
|          2|  Alice |       2000|   2|
|          1|John Doe|       1000|   3|
|          3|    NULL|       -500|   4|
+-----------+--------+-----------+----+



### LAG

In [0]:
window_lag = Window.partitionBy("customer_id").orderBy("order_date")
lag_df = inner_df.withColumn("prev_order_amount",lag("amount", 1).over(window_lag))
lag_df.show()

+-----------+--------+----------+------+--------+-----------------+---------+-----------------+
|customer_id|order_id|order_date|amount|    name|            email|     city|prev_order_amount|
+-----------+--------+----------+------+--------+-----------------+---------+-----------------+
|          1|     101|2024-01-01|  1000|John Doe| john@example.com|Hyderabad|             NULL|
|          1|     105|2024-01-05|  NULL|John Doe| john@example.com|Hyderabad|             1000|
|          2|     102|2024-01-02|  2000|  Alice |alice@example.com|  Chennai|             NULL|
|          3|     103|2024-01-03|  -500|    NULL|  bob@example.com|Bangalore|             NULL|
|          5|     106|2024-01-06|  3000|     Eva|  eva@example.com|Hyderabad|             NULL|
|          5|     107|2024-01-07|  3000|     Eva|  eva@example.com|Hyderabad|             3000|
+-----------+--------+----------+------+--------+-----------------+---------+-----------------+



### Practice Set C: Date Analysis

### Extract Month

In [0]:
orders_with_date = orders.withColumn("order_date",to_date("order_date"))
orders_with_date = orders_with_date.withColumn("month",month("order_date"))
orders_with_date.show()

+--------+-----------+----------+------+-----+
|order_id|customer_id|order_date|amount|month|
+--------+-----------+----------+------+-----+
|     101|          1|2024-01-01|  1000|    1|
|     102|          2|2024-01-02|  2000|    1|
|     103|          3|2024-01-03|  -500|    1|
|     104|         99|2024-01-04|  1500|    1|
|     105|          1|2024-01-05|  NULL|    1|
|     106|          5|2024-01-06|  3000|    1|
|     107|          5|2024-01-07|  3000|    1|
+--------+-----------+----------+------+-----+



### Monthly Sales Aggregation

In [0]:
monthly_sales = orders_with_date.groupBy("month").agg(sum("amount").alias("total_sales"))
monthly_sales.show()

+-----+-----------+
|month|total_sales|
+-----+-----------+
|    1|      10000|
+-----+-----------+



### Date Difference

In [0]:
date_diff_df = orders_with_date.withColumn("days_diff",datediff(current_date(), col("order_date")))
date_diff_df.show()

+--------+-----------+----------+------+-----+---------+
|order_id|customer_id|order_date|amount|month|days_diff|
+--------+-----------+----------+------+-----+---------+
|     101|          1|2024-01-01|  1000|    1|      832|
|     102|          2|2024-01-02|  2000|    1|      831|
|     103|          3|2024-01-03|  -500|    1|      830|
|     104|         99|2024-01-04|  1500|    1|      829|
|     105|          1|2024-01-05|  NULL|    1|      828|
|     106|          5|2024-01-06|  3000|    1|      827|
|     107|          5|2024-01-07|  3000|    1|      826|
+--------+-----------+----------+------+-----+---------+



### Monthly Trend

In [0]:
monthly_trend = orders_with_date.groupBy("month").agg(sum("amount").alias("total_sales"),count("*").alias("order_count")).orderBy("month")
monthly_trend.show()

+-----+-----------+-----------+
|month|total_sales|order_count|
+-----+-----------+-----------+
|    1|      10000|          7|
+-----+-----------+-----------+



### Practice Set D: Timed Pipeline

### Clean Invalid Records

In [0]:
clean_orders = orders.filter((col("amount").isNotNull()) & (col("amount") > 0))
clean_customers = customers.filter(col("name").isNotNull() & col("email").isNotNull())

### Validate Referential Integrity

In [0]:
invalid_fk = clean_orders.join(clean_customers, on="customer_id", how="left_anti")
invalid_fk.show()

+-----------+--------+----------+------+
|customer_id|order_id|order_date|amount|
+-----------+--------+----------+------+
|         99|     104|2024-01-04|  1500|
+-----------+--------+----------+------+



### Join Tables

In [0]:
joined_df = clean_orders.join(clean_customers, on="customer_id", how="inner")

### Aggregations + Windows

In [0]:
# total spend per customer
agg_df = joined_df.groupBy("customer_id", "name", "city").agg(sum("amount").alias("total_spend"))
# rank within city
window_city_rank = Window.partitionBy("city").orderBy(col("total_spend").desc())
final_df = agg_df.withColumn("city_rank",dense_rank().over(window_city_rank)).display()

customer_id,name,city,total_spend,city_rank
2,Alice,Chennai,2000,1
5,Eva,Hyderabad,6000,1
1,John Doe,Hyderabad,1000,2
